# Debug Trading Environment Episode

Run an episode and visualize actions, rewards, positions, and portfolio value.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from numpy.ma.core import product
from rl_trading_lab.environment.trading_env import TradingEnv
from rl_trading_lab.environment import Action
from rl_trading_lab.utils.data_processor import DataProcessor
from rl_trading_lab.utils import CheckpointManager
from rl_trading_lab.config import load_config
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir
from stable_baselines3.common import vec_env

from test_portfolio_value_bug import truncated, terminated, reward

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Imports successful!")

Imports successful!


In [2]:
import os; os.chdir("..")

In [3]:
print(os.getcwd())

/Users/mohamedali/trading_project/rl-trading-lab


## Load Configuration and Data

In [4]:
# Load config using Hydra
config_dir = Path('configs').resolve()

with initialize_config_dir(config_dir=str(config_dir), version_base=None):
    cfg = compose(config_name='config', overrides=[])
    config = load_config(cfg)

print(f"Reward type: {config.env.environment_params.reward_type}")
print(f"Initial balance: ${config.env.environment_params.initial_balance:,.2f}")
print(f"Commission rate: {config.env.environment_params.commission_rate}")
print(f"Lookback window: {config.env.environment_params.lookback_window}")
print(f"Hold closes position: {config.env.environment_params.hold_closes_position}")

Reward type: returns
Initial balance: $10,000.00
Commission rate: 0.0
Lookback window: 20
Hold closes position: True


In [5]:
# Load data using DataProcessor
data_processor = DataProcessor(
    data_path=config.data.train_data_path,
    observation_config=config.observation,
    feature_engineering_config=config.feature_engineering,
    val_split=config.data.val_split,
    test_split=config.data.test_split,
)

train_df, val_df, test_df, observation_features = data_processor.process()

print(f"Train: {len(train_df)} bars")
print(f"Val: {len(val_df)} bars")
print(f"Test: {len(test_df)} bars")
print(f"Observation features ({len(observation_features)}): {observation_features}")
print(f"\nAvailable columns in data: {list(train_df.columns)[:10]}...")

Train: 27585 bars
Val: 7881 bars
Test: 3940 bars
Observation features (4): ['ratio_sma_5_close_zscore', 'ratio_sma_20_close_zscore', 'ratio_range_close_zscore', 'fracdiff_0.4_zscore']

Available columns in data: ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'ratio_sma_5_close', 'ratio_sma_20_close', 'ratio_range_close', 'fracdiff_0.4']...


## Create Environment

In [6]:
df = train_df

In [7]:
# Create test environment
env_params = config.env.environment_params

test_env = TradingEnv(
    df=df,
    lookback_window=env_params.lookback_window,
    initial_balance=env_params.initial_balance,
    commission_rate=env_params.commission_rate,
    slippage_rate=env_params.slippage_rate,
    reward_type=env_params.reward_type,
    discrete_actions=env_params.discrete_actions,
    max_position_pct=env_params.max_position_pct,
    features_to_use=observation_features,  # Use observation_features instead of feature_names
    randomize_start=False,
    min_episode_length=env_params.min_episode_length,
    hold_closes_position=env_params.hold_closes_position,
    price_column=config.env.price_column,
)

print(f"✓ Environment created")
print(f"  Observation space: {test_env.observation_space.shape}")
print(f"  Action space: {test_env.action_space}")
print(f"  Max steps: {test_env.max_steps}")
print(f"  Hold closes position: {test_env.hold_closes_position}")

✓ Environment created
  Observation space: (83,)
  Action space: Discrete(3)
  Max steps: 27584
  Hold closes position: True


## Load Trained Agent with CheckpointManager

In [8]:
# Load latest trained model using CheckpointManager
# Set to None
# for latest, or specify a run name
run_id = None or "A2C_returns_20251028_143912"
checkpoint_dir = Path("checkpoints")

if run_id and (checkpoint_dir / run_id).exists():
    # Use specified run_id
    latest_run = checkpoint_dir / run_id
    print(f"📂 Loading from specified run: {latest_run.name}\n")
else:
    # Find latest training run automatically
    training_runs = [d for d in checkpoint_dir.iterdir() if d.is_dir()]
    training_runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    latest_run = training_runs[0]
    print(f"📂 Loading from latest run: {latest_run.name}\n")

# Load with CheckpointManager
manager = CheckpointManager(latest_run)
model, test_vec_env = manager.load_best_model(test_env, verbose=1)

print(f"\n✓ Model loaded successfully!")
print(f"  Policy: {type(model.policy).__name__}")
print(f"  Device: {model.device}")

📂 Loading from specified run: A2C_returns_20251028_143912


✓ Model loaded successfully!
  Policy: ActorCriticPolicy
  Device: cpu


In [16]:
test_env.action_space.sample()

np.int64(1)

In [9]:
type(model)

stable_baselines3.a2c.a2c.A2C

In [21]:
obs = test_env.reset()
rewards = []
actions = []
for i in range(1000):
    action = test_env.action_space.sample()
    obs, reward, terminated, truncated, info = test_env.step(action)
    done = terminated or truncated
    actions.append(action)
    rewards.append(reward)
    if done:
        break

In [20]:
obs.shape

(83,)

In [28]:
import math
math.prod(1 + r for r in rewards) - 1

np.float64(0.025093769569370927)

In [30]:
sum(rewards)

np.float64(0.02484713892541854)

In [31]:
info

{'step': 1020,
 'balance': np.float64(10250.937695693752),
 'portfolio_value': np.float64(10250.937695693752),
 'position': np.float64(0.08602767820110133),
 'total_return': np.float64(0.02509376956937522),
 'num_trades': 452,
 'sharpe': np.float64(1.1134372437607631),
 'max_drawdown': np.float64(0.005000121408992337)}

In [32]:
0.025093769569370927 / 452 * 10_000

0.5551718931276755

## Run Episode with Trained Agent

In [ ]:
def run_agent_episode(vec_env, model):
    """Run episode with trained agent"""
    obs = vec_env.reset()
    
    # Get the unwrapped environment for direct access
    # Need to unwrap: VecNormalize -> DummyVecEnv -> Monitor -> TradingEnv
    unwrapped_env = vec_env.venv.envs[0].env
    
    data = {
        'step': [],
        'action': [],
        'reward': [],
        'position': [],
        'balance': [],
        'portfolio_value': [],
        'price': [],
        'terminated': [],
    }
    
    done = False
    step = 0
    
    while not done:
        # Get action from agent
        action, _ = model.predict(obs, deterministic=True)
        
        # Step
        obs, reward, done, info = vec_env.step(action)
        
        # Extract from vectorized format
        if isinstance(done, np.ndarray):
            done = done[0]
        if isinstance(action, np.ndarray):
            action = action[0]
        if isinstance(reward, np.ndarray):
            reward = reward[0]
        
        # Collect data from unwrapped env
        data['step'].append(step)
        data['action'].append(action)
        data['reward'].append(reward)
        data['position'].append(unwrapped_env.position.size)
        data['balance'].append(unwrapped_env.balance)
        data['portfolio_value'].append(unwrapped_env._get_portfolio_value())
        data['price'].append(unwrapped_env._get_current_price())
        data['terminated'].append(done)
        
        step += 1
        
        if step > 500:  # Safety limit
            break
    
    return pd.DataFrame(data)

# Run episode
df_agent = run_agent_episode(test_vec_env, model)

print(f"\n{'='*60}")
print(f"Episode completed: {len(df_agent)} steps")
print(f"Final portfolio value: ${df_agent['portfolio_value'].iloc[-1]:,.2f}")
print(f"Final return: {(df_agent['portfolio_value'].iloc[-1] / env_params.initial_balance - 1) * 100:.2f}%")
print(f"Final position: {df_agent['position'].iloc[-1]:.4f}")
print(f"{'='*60}")

df_agent.head()

---

In [ ]:
inputs = np.random.random((84,))
inputs
import torch
torch.from_numpy(inputs)

In [ ]:
model.policy(torch.from_numpy(inputs))

In [ ]:
model.predict(inputs, deterministic=True)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

# 1. Portfolio Value
axes[0].plot(df_agent['step'], df_agent['portfolio_value'], label='Portfolio Value', linewidth=2)
axes[0].axhline(y=env_params.initial_balance, color='red', linestyle='--', alpha=0.5, label='Initial Balance')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title('Portfolio Value Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Price
axes[1].plot(df_agent['step'], df_agent['price'], label='Price', color='orange', linewidth=1.5)
axes[1].set_ylabel('Price ($)')
axes[1].set_title('Asset Price')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Actions - Use Action enum for clarity
action_colors = {Action.HOLD: 'gray', Action.BUY: 'green', Action.SELL: 'red'}
action_labels = {Action.HOLD: 'Hold', Action.BUY: 'Buy', Action.SELL: 'Sell'}
for action_val in [Action.HOLD, Action.BUY, Action.SELL]:
    mask = df_agent['action'] == action_val
    axes[2].scatter(df_agent[mask]['step'], df_agent[mask]['action'], 
                   c=action_colors[action_val], label=action_labels[action_val], alpha=0.6, s=20)
axes[2].set_ylabel('Action')
axes[2].set_title('Actions Taken')
axes[2].set_yticks([Action.HOLD, Action.BUY, Action.SELL])
axes[2].set_yticklabels(['Hold', 'Buy', 'Sell'])
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# 4. Position
axes[3].plot(df_agent['step'], df_agent['position'], label='Position Size', color='purple', linewidth=1.5)
axes[3].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[3].set_ylabel('Position Size')
axes[3].set_title('Position Size Over Time')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

# 5. Rewards
axes[4].plot(df_agent['step'], df_agent['reward'], label='Reward', color='blue', alpha=0.7, linewidth=1)
axes[4].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[4].set_ylabel('Reward')
axes[4].set_xlabel('Step')
axes[4].set_title('Rewards')
axes[4].legend()
axes[4].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Analysis: Check for Issues

In [ ]:
print("=" * 60)
print("EPISODE SUMMARY")
print("=" * 60)
print(f"Total steps: {len(df_agent)}")
print(f"Initial balance: ${env_params.initial_balance:,.2f}")
print(f"Final portfolio value: ${df_agent['portfolio_value'].iloc[-1]:,.2f}")
print(f"Final return: {(df_agent['portfolio_value'].iloc[-1] / env_params.initial_balance - 1) * 100:.2f}%")
print()

print("Action Distribution:")
print(df_agent['action'].value_counts().sort_index())
print()

print("Reward Statistics:")
print(f"  Mean: {df_agent['reward'].mean():.4f}")
print(f"  Std: {df_agent['reward'].std():.4f}")
print(f"  Min: {df_agent['reward'].min():.4f}")
print(f"  Max: {df_agent['reward'].max():.4f}")
print()

print("Position Statistics:")
print(f"  Final position: {df_agent['position'].iloc[-1]:.4f}")
print(f"  Max position: {df_agent['position'].max():.4f}")
print(f"  Min position: {df_agent['position'].min():.4f}")
print(f"  % of time in position: {(df_agent['position'] != 0).sum() / len(df_agent) * 100:.1f}%")
print()

# Check if position is open at end
if abs(df_agent['position'].iloc[-1]) > 0.01:
    print("⚠️  WARNING: Position is still OPEN at end of episode!")
    print(f"   Open position size: {df_agent['position'].iloc[-1]:.4f}")
    print(f"   This unrealized P&L is NOT reflected in the return calculation!")
    print()
    
    # Calculate what the return would be if we closed the position
    unwrapped_env = test_env
    current_price = df_agent['price'].iloc[-1]
    position_size = df_agent['position'].iloc[-1]
    
    if hasattr(unwrapped_env, 'position') and unwrapped_env.position.entry_price > 0:
        entry_price = unwrapped_env.position.entry_price
        unrealized_pnl = position_size * (current_price - entry_price)
        print(f"   Entry price: ${entry_price:.2f}")
        print(f"   Current price: ${current_price:.2f}")
        print(f"   Unrealized P&L: ${unrealized_pnl:.2f}")
else:
    print("✓ Position is FLAT at end of episode (good!)")
    print()

## Deep Dive: Balance vs Portfolio Value

In [ ]:
# Calculate what balance + position value should equal
df_agent['position_value'] = abs(df_agent['position']) * df_agent['price']
df_agent['balance_plus_position'] = df_agent['balance'] + df_agent['position_value']

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Balance
axes[0].plot(df_agent['step'], df_agent['balance'], label='Balance (Cash)', linewidth=2)
axes[0].set_ylabel('Balance ($)')
axes[0].set_title('Cash Balance Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Position Value
axes[1].plot(df_agent['step'], df_agent['position_value'], label='Position Value', color='orange', linewidth=2)
axes[1].set_ylabel('Position Value ($)')
axes[1].set_title('Position Value Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Comparison
axes[2].plot(df_agent['step'], df_agent['portfolio_value'], label='Portfolio Value (from env)', linewidth=2)
axes[2].plot(df_agent['step'], df_agent['balance_plus_position'], 
            label='Balance + Position Value', linestyle='--', linewidth=2, alpha=0.7)
axes[2].set_ylabel('Value ($)')
axes[2].set_xlabel('Step')
axes[2].set_title('Portfolio Value: Environment vs Manual Calculation')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Check if they match
diff = (df_agent['portfolio_value'] - df_agent['balance_plus_position']).abs().max()
print(f"Max difference between env.portfolio_value and balance+position: ${diff:.2f}")
if diff > 1.0:
    print("⚠️  Large discrepancy detected! This suggests an accounting bug.")
else:
    print("✓ Values match (within $1 tolerance)")

In [ ]:
# Trade distribution analysis
if len(trade_history) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # 1. P&L Distribution
    axes[0, 0].hist(df_trades['net_pnl'], bins=30, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Break-even')
    axes[0, 0].axvline(x=df_trades['net_pnl'].mean(), color='green', linestyle='--', linewidth=2, label='Mean')
    axes[0, 0].set_xlabel('Net P&L ($)', fontsize=11)
    axes[0, 0].set_ylabel('Frequency', fontsize=11)
    axes[0, 0].set_title('P&L Distribution', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Return % Distribution
    axes[0, 1].hist(df_trades['return_pct'] * 100, bins=30, alpha=0.7, color='purple', edgecolor='black')
    axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('Return (%)', fontsize=11)
    axes[0, 1].set_ylabel('Frequency', fontsize=11)
    axes[0, 1].set_title('Return % Distribution', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Hold Duration Distribution
    axes[1, 0].hist(df_trades['hold_bars'], bins=30, alpha=0.7, color='orange', edgecolor='black')
    axes[1, 0].set_xlabel('Hold Duration (bars)', fontsize=11)
    axes[1, 0].set_ylabel('Frequency', fontsize=11)
    axes[1, 0].set_title('Hold Duration Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Win/Loss by Side
    if 'side' in df_trades.columns:
        side_pnl = df_trades.groupby('side')['net_pnl'].agg(['sum', 'mean', 'count'])
        x = np.arange(len(side_pnl))
        width = 0.35
        
        axes[1, 1].bar(x - width/2, side_pnl['sum'], width, label='Total P&L', alpha=0.7)
        axes[1, 1].bar(x + width/2, side_pnl['mean'] * side_pnl['count'], width, label='Avg P&L × Count', alpha=0.7)
        
        axes[1, 1].set_xlabel('Side', fontsize=11)
        axes[1, 1].set_ylabel('P&L ($)', fontsize=11)
        axes[1, 1].set_title('P&L by Side (Long vs Short)', fontsize=12, fontweight='bold')
        axes[1, 1].set_xticks(x)
        axes[1, 1].set_xticklabels(side_pnl.index)
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize trades on price chart
if len(trade_history) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
    
    # Plot price
    axes[0].plot(df_agent['step'], df_agent['price'], label='Price', color='black', linewidth=1.5, alpha=0.7)
    
    # Plot entry and exit points
    for _, trade in df_trades.iterrows():
        color = 'green' if trade['side'] == 'LONG' else 'red'
        marker_open = '^' if trade['side'] == 'LONG' else 'v'
        marker_close = 'v' if trade['side'] == 'LONG' else '^'
        
        # Entry point
        axes[0].scatter(trade['open_step'], trade['entry_price'], 
                       c=color, marker=marker_open, s=100, alpha=0.7, 
                       edgecolors='black', linewidths=1)
        
        # Exit point
        axes[0].scatter(trade['close_step'], trade['exit_price'], 
                       c=color, marker=marker_close, s=100, alpha=0.7,
                       edgecolors='black', linewidths=1)
        
        # Connect with line
        axes[0].plot([trade['open_step'], trade['close_step']], 
                    [trade['entry_price'], trade['exit_price']], 
                    color=color, alpha=0.3, linewidth=1)
    
    axes[0].set_ylabel('Price ($)', fontsize=12)
    axes[0].set_title('Trade Entry/Exit Points on Price Chart', fontsize=14, fontweight='bold')
    axes[0].legend(['Price', 'Long Entry (▲)', 'Long Exit (▼)', 'Short Entry (▼)', 'Short Exit (▲)'], 
                   loc='best')
    axes[0].grid(True, alpha=0.3)
    
    # Plot cumulative P&L
    df_trades['cumulative_pnl'] = df_trades['net_pnl'].cumsum()
    axes[1].plot(df_trades['close_step'], df_trades['cumulative_pnl'], 
                label='Cumulative P&L', color='blue', linewidth=2)
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
    axes[1].fill_between(df_trades['close_step'], 0, df_trades['cumulative_pnl'], 
                        where=df_trades['cumulative_pnl'] >= 0, color='green', alpha=0.3)
    axes[1].fill_between(df_trades['close_step'], 0, df_trades['cumulative_pnl'], 
                        where=df_trades['cumulative_pnl'] < 0, color='red', alpha=0.3)
    
    axes[1].set_ylabel('Cumulative P&L ($)', fontsize=12)
    axes[1].set_xlabel('Step', fontsize=12)
    axes[1].set_title('Cumulative P&L Over Time', fontsize=14, fontweight='bold')
    axes[1].legend(loc='best')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Plotted {len(df_trades)} trades")

In [ ]:
# Get trade history from environment
unwrapped_env = test_vec_env.venv.envs[0].env
trade_history = unwrapped_env.get_trade_history()

print(f"Total completed trades: {len(trade_history)}\n")

if len(trade_history) > 0:
    # Convert to DataFrame
    df_trades = pd.DataFrame(trade_history)
    
    # Display first few trades
    print("First 10 trades:")
    display(df_trades.head(10))
    
    # Trade statistics
    print("\n" + "="*60)
    print("TRADE STATISTICS")
    print("="*60)
    
    # Win/Loss analysis
    winning_trades = df_trades[df_trades['net_pnl'] > 0]
    losing_trades = df_trades[df_trades['net_pnl'] < 0]
    
    print(f"Total trades: {len(df_trades)}")
    print(f"Winning trades: {len(winning_trades)} ({len(winning_trades)/len(df_trades)*100:.1f}%)")
    print(f"Losing trades: {len(losing_trades)} ({len(losing_trades)/len(df_trades)*100:.1f}%)")
    print()
    
    # P&L statistics
    print(f"Total P&L: ${df_trades['net_pnl'].sum():.2f}")
    print(f"Average P&L per trade: ${df_trades['net_pnl'].mean():.2f}")
    print(f"Best trade: ${df_trades['net_pnl'].max():.2f}")
    print(f"Worst trade: ${df_trades['net_pnl'].min():.2f}")
    print()
    
    if len(winning_trades) > 0:
        print(f"Average winning trade: ${winning_trades['net_pnl'].mean():.2f}")
    if len(losing_trades) > 0:
        print(f"Average losing trade: ${losing_trades['net_pnl'].mean():.2f}")
    
    # Profit factor
    if len(losing_trades) > 0 and losing_trades['net_pnl'].sum() < 0:
        profit_factor = abs(winning_trades['net_pnl'].sum() / losing_trades['net_pnl'].sum())
        print(f"Profit factor: {profit_factor:.2f}")
    print()
    
    # Return statistics
    print(f"Average return per trade: {df_trades['return_pct'].mean()*100:.2f}%")
    print(f"Best return: {df_trades['return_pct'].max()*100:.2f}%")
    print(f"Worst return: {df_trades['return_pct'].min()*100:.2f}%")
    print()
    
    # Hold duration
    print(f"Average hold duration: {df_trades['hold_bars'].mean():.1f} bars")
    print(f"Median hold duration: {df_trades['hold_bars'].median():.0f} bars")
    print(f"Min hold: {df_trades['hold_bars'].min()} bars")
    print(f"Max hold: {df_trades['hold_bars'].max()} bars")
    print()
    
    # Long vs Short
    long_trades = df_trades[df_trades['side'] == 'LONG']
    short_trades = df_trades[df_trades['side'] == 'SHORT']
    print(f"Long trades: {len(long_trades)} ({len(long_trades)/len(df_trades)*100:.1f}%)")
    print(f"Short trades: {len(short_trades)} ({len(short_trades)/len(df_trades)*100:.1f}%)")
    
else:
    print("No trades completed in this episode")

## Trade History Analysis